In [30]:
import os
import sympy as sp
import numpy as np
from IPython.display import display
import pandas as pd

In [34]:
PARAMS_DIR = os.path.join(os.getcwd(), "../mdh_params")

In [26]:
# Define the symbolic variables
alpha, a, theta, d = sp.symbols('alpha a theta d')

In [55]:
def mdh_tf(alpha, a, theta, d):
    """
    Compute the Modified Denavit-Hartenberg transformation matrix.

    Parameters:
    alpha : float
        The twist angle (in radians).
    a : float
        The link length.
    theta : float
        The joint angle (in radians).
    d : float
        The link offset.

    Returns:
    tf : sympy.Matrix
        The 4x4 transformation matrix.
    """

    tf = sp.Matrix([
        [sp.cos(theta),               -sp.sin(theta),              0,               a               ],   # type: ignore
        [sp.sin(theta)*sp.cos(alpha), sp.cos(theta)*sp.cos(alpha), -sp.sin(alpha),  -d*sp.sin(alpha)],   # type: ignore
        [sp.sin(alpha)*sp.sin(theta), sp.sin(alpha)*sp.cos(theta), sp.cos(alpha),   d*sp.cos(alpha) ],   # type: ignore
        [0,                           0,                           0,               1               ]
    ])
    
    return tf

In [64]:
param_files = os.listdir(PARAMS_DIR)
display(param_files)

['scara.csv',
 'cylindrical.csv',
 'spherical.csv',
 'articulated.csv',
 'cartesian.csv']

In [88]:
def parse_angle_deg(val):
    """Converts numbers or string expressions in degrees to SymPy radians."""
    # Parse strings (like 'theta1' or 'theta1 + 180') or keep numbers/symbols as-is
    expr = sp.sympify(val)
    
    # If it's a pure number (int/float), convert degrees to radians
    if expr.is_number:
        return sp.rad(expr)
    
    # If it's a symbolic expression (e.g., theta1 + 90), convert any numeric constants to radians
    # Note: If your expressions are just 'theta1', this leaves 'theta1' untouched as a symbol
    return expr.replace(lambda x: x.is_Number and x != 0, lambda x: sp.rad(x))

In [95]:
def compute_mdh_transformations(mdh_params):
    """
    Compute the MDH transformation matrices for a given parameter file.

    Parameters:
    mdh_params : DataFrame
        The DataFrame containing the MDH parameters.

    Returns:
    transformations : list of sympy.Matrix
        A list of 4x4 transformation matrices for each link.
    """

    transformations = []
    
    for index, row in mdh_params.iterrows():
        alpha_val = parse_angle_deg(row['alpha'])
        a_val = sp.sympify(row['a'])
        theta_val = parse_angle_deg(row['theta'])
        d_val = sp.sympify(row['d'])
        
        tf = mdh_tf(alpha_val, a_val, theta_val, d_val)
        transformations.append(tf)

        # Display each transformation matrix
        label = sp.MatrixSymbol(f'A{index+1}', 4, 4)
        eq = sp.Eq(label, tf)
        display(eq)
    
    return transformations

In [101]:
for param_file in param_files:
    manipulator_name = param_file.split('.')[0]
    print(f"Manipulator: {manipulator_name.capitalize()}")
    mdh_params = pd.read_csv(f"{PARAMS_DIR}/{param_file}")
    display(mdh_params)

    tf_matrices = compute_mdh_transformations(mdh_params)

    T_0_n = sp.eye(4)  # Initialize the overall transformation matrix as identity
    for i, tf in enumerate(tf_matrices):
        T_0_n = sp.simplify(T_0_n * tf)  # Multiply the transformation matrices
    
    label = sp.MatrixSymbol(f'{{}}^{{{0}}}T_{{{i+1}}}', 4, 4)
    eq = sp.Eq(label, T_0_n)
    display(eq)
    print("=" * 100 + "\n" * 2)


Manipulator: Scara


,alpha,a,theta,d
0,0,0,theta1,d1
1,0,a2,theta2,0
2,0,a3,0,0
3,180,0,0,d4


Eq(A1, Matrix([
[cos(theta1), -sin(theta1), 0,  0],
[sin(theta1),  cos(theta1), 0,  0],
[          0,            0, 1, d1],
[          0,            0, 0,  1]]))

Eq(A2, Matrix([
[cos(theta2), -sin(theta2), 0, a2],
[sin(theta2),  cos(theta2), 0,  0],
[          0,            0, 1,  0],
[          0,            0, 0,  1]]))

Eq(A3, Matrix([
[1, 0, 0, a3],
[0, 1, 0,  0],
[0, 0, 1,  0],
[0, 0, 0,  1]]))

Eq(A4, Matrix([
[1,  0,  0,   0],
[0, -1,  0,   0],
[0,  0, -1, -d4],
[0,  0,  0,   1]]))

... (19 outputs left out)

Eq({}^{0}T_{4}, Matrix([
[cos(theta1)*cos(theta2), -sin(theta2)*cos(theta1), -sin(theta1),     (a2 + a3)*cos(theta1)*cos(theta2)],
[sin(theta1)*cos(theta2), -sin(theta1)*sin(theta2),  cos(theta1),     (a2 + a3)*sin(theta1)*cos(theta2)],
[           -sin(theta2),             -cos(theta2),            0, -a2*sin(theta2) - a3*sin(theta2) + d1],
[                      0,                        0,            0,                                     1]]))



Manipulator: Cartesian


,alpha,a,theta,d


Eq({}^{0}T_{4}, Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]]))

In [102]:
!jupyter nbconvert --to markdown manipulator_mdh.ipynb


[NbConvertApp] Converting notebook manipulator_mdh.ipynb to markdown
[NbConvertApp] Writing 13262 bytes to manipulator_mdh.md
